# Kapanin — EDA, prioritas stok, dan training awal

**Pertanyaan bisnis:** produk mana yang stoknya perlu ditinjau, bagaimana pola penjualannya, dan data apa yang diperlukan sebelum merekomendasikan diskon? Notebook ini memakai satu toko dengan data sintetis. Bagian 1–6 adalah eksplorasi benchmark; bagian 7 membuat prioritas dari field backend; bagian 8 melatih dan mengevaluasi prediksi permintaan mingguan.

Grafik dan nilai risiko di bagian eksplorasi bukan hasil bisnis toko nyata. Notebook ini tidak mengestimasi efek kausal diskon atau persentase diskon optimal.


## 0. Muat data

Di repo lokal, notebook mencari `data/store_1`. Di Colab, notebook membaca CSV dari branch `main` GitHub. Pastikan ketiga file sudah tersedia dan repo dapat diakses publik. Jalankan seluruh sel dari atas ke bawah. Kebutuhan inti: Pandas dan NumPy; Matplotlib hanya untuk grafik.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

candidates = [
    Path.cwd() / "data" / "store_1",
    Path.cwd() / "Kapanin" / "data" / "store_1",
    Path.cwd().parent / "data" / "store_1",
]
required_files = ("stores.csv", "products.csv", "transactions.csv")
DATA_DIR = next((path for path in candidates if all((path / name).exists() for name in required_files)), None)
RAW_BASE_URL = "https://raw.githubusercontent.com/faizwibawa/Kapanin/main/data/store_1"

def csv_source(filename):
    return DATA_DIR / filename if DATA_DIR is not None else f"{RAW_BASE_URL}/{filename}"

print("Sumber data:", DATA_DIR.resolve() if DATA_DIR is not None else RAW_BASE_URL)

try:
    import matplotlib.pyplot as plt
    HAS_PLOTS = True
    plt.rcParams.update({
        "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
        "axes.titlesize": 11, "axes.titleweight": "bold", "font.size": 9,
    })
except ImportError:
    HAS_PLOTS = False
    print("Matplotlib tidak tersedia. Di Colab grafik akan tampil setelah Matplotlib tersedia.")


In [ ]:
stores = pd.read_csv(csv_source("stores.csv"), dtype={"store_id": "string"})
products = pd.read_csv(
    csv_source("products.csv"),
    dtype={"product_id": "string", "store_id": "string", "sku": "string"},
    parse_dates=["stock_as_of_date", "stock_received_date", "expiry_date"],
)
transactions = pd.read_csv(
    csv_source("transactions.csv"),
    dtype={"transaction_id": "string", "store_id": "string", "product_id": "string"},
    parse_dates=["transaction_date"],
)
snapshot = products.stock_as_of_date.max()
print(f"Toko: {stores.store_name.iloc[0]} | snapshot stok: {snapshot.date()}")
print(f"Produk: {len(products):,} | transaksi: {len(transactions):,}")
print(f"Periode transaksi: {transactions.transaction_date.min().date()} s.d. {transactions.transaction_date.max().date()}")


## 1. Pemeriksaan kualitas data

Pemeriksaan ini memisahkan masalah data yang dapat mengubah keputusan: kunci ganda, transaksi tanpa produk, nilai harga tidak masuk akal, tanggal transaksi setelah snapshot, dan stok yang tanggal kedaluwarsanya sudah tiba. Jumlah temuan ditampilkan; analisis sebaiknya ditahan bila ada masalah pada kunci atau harga.


In [ ]:
quality_checks = {
    "duplikat_store_id": stores.store_id.duplicated().sum(),
    "duplikat_product_id": products.product_id.duplicated().sum(),
    "duplikat_transaction_id": transactions.transaction_id.duplicated().sum(),
    "transaksi_tanpa_produk": (~transactions.product_id.isin(products.product_id)).sum(),
    "produk_tanpa_toko": (~products.store_id.isin(stores.store_id)).sum(),
    "transaksi_tanpa_toko": (~transactions.store_id.isin(stores.store_id)).sum(),
    "stok_negatif": products.stock_on_hand.lt(0).sum(),
    "harga_atau_biaya_tidak_positif": (products.list_price.le(0) | products.unit_cost.le(0)).sum(),
    "harga_jual_tidak_positif": transactions.unit_sale_price.le(0).sum(),
    "diskon_di_luar_0_sampai_1": (~transactions.discount_rate.between(0, 1)).sum(),
    "transaksi_setelah_snapshot": transactions.transaction_date.ge(snapshot).sum(),
    "stok_positif_kedaluwarsa_hari_ini_atau_lalu": ((products.stock_on_hand > 0) & (products.expiry_date <= snapshot)).sum(),
}
quality_report = pd.Series(quality_checks, name="jumlah_temuan").to_frame()
print(quality_report.to_string())
assert quality_report.loc[
    ["duplikat_store_id", "duplikat_product_id", "duplikat_transaction_id", "transaksi_tanpa_produk",
     "produk_tanpa_toko", "transaksi_tanpa_toko", "stok_negatif", "harga_atau_biaya_tidak_positif",
     "harga_jual_tidak_positif", "diskon_di_luar_0_sampai_1", "transaksi_setelah_snapshot"],
    "jumlah_temuan"
].eq(0).all(), "Ada masalah data inti; periksa quality_report sebelum melanjutkan."


## 2. Kinerja penjualan

Ukur unit dan omzet berdasarkan **harga yang benar-benar dibayar**. Lihat tren mingguan serta kategori dengan kontribusi terbesar. Tren historis membantu memahami skala aktivitas, tetapi tidak boleh langsung ditafsirkan sebagai efek dari diskon atau stok.


In [ ]:
sales = transactions.merge(
    products[["product_id", "category", "unit_cost"]], on="product_id", how="left", validate="many_to_one"
)
sales["revenue"] = sales.quantity * sales.unit_sale_price
sales["week_start"] = sales.transaction_date - pd.to_timedelta(sales.transaction_date.dt.weekday, unit="D")
weekly_sales = sales.groupby("week_start", as_index=False).agg(
    units_sold=("quantity", "sum"), revenue=("revenue", "sum"), transactions=("transaction_id", "count")
)
# Hanya minggu Senin–Minggu penuh; minggu pertama/terakhir bisa parsial.
first_full_monday = transactions.transaction_date.min() + pd.Timedelta(
    days=(-transactions.transaction_date.min().weekday()) % 7
)
last_date = transactions.transaction_date.max()
weekly_sales = weekly_sales.loc[
    (weekly_sales.week_start >= first_full_monday) &
    (weekly_sales.week_start + pd.Timedelta(days=6) <= last_date)
].reset_index(drop=True)
category_sales = sales.groupby("category", as_index=False).agg(
    units_sold=("quantity", "sum"), revenue=("revenue", "sum"), products_sold=("product_id", "nunique")
).sort_values("revenue", ascending=False)
category_sales["revenue_share"] = category_sales.revenue / category_sales.revenue.sum()
print("Penjualan mingguan terakhir:")
print(weekly_sales.tail(8).to_string(index=False))
print("\n10 kategori dengan omzet tertinggi:")
print(category_sales.head(10).to_string(index=False, formatters={"revenue_share": "{:.1%}".format}))


In [ ]:
# Grafik 1–3: tren unit, omzet, dan kontribusi kategori.
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    weekly_view = weekly_sales.copy()
    weekly_view["avg_4w"] = weekly_view.units_sold.rolling(4, min_periods=4).mean()
    axes[0].plot(weekly_view.week_start, weekly_view.units_sold, color="#9ecae1", linewidth=1, label="Mingguan")
    axes[0].plot(weekly_view.week_start, weekly_view.avg_4w, color="#0b6e75", linewidth=2, label="Rata-rata 4 minggu")
    axes[0].set(title="Unit terjual per minggu penuh", ylabel="Unit")
    axes[0].legend(frameon=False)
    axes[1].plot(weekly_view.week_start, weekly_view.revenue / 1_000_000, color="#e07a3f")
    axes[1].set(title="Omzet per minggu penuh", ylabel="Juta rupiah")
    top_categories = category_sales.head(10).sort_values("revenue")
    axes[2].barh(top_categories.category, top_categories.revenue / 1_000_000, color="#367c8d")
    axes[2].set(title="10 kategori dengan omzet tertinggi", xlabel="Juta rupiah")
    for ax in axes[:2]:
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=.2)
    axes[2].grid(axis="x", alpha=.2)
    fig.tight_layout()
    plt.show()
print(f"Insight: kategori omzet terbesar adalah {category_sales.iloc[0].category} "
      f"({category_sales.iloc[0].revenue_share:.1%} dari omzet). "
      "Periksa perubahan tren mingguan bersama konteks stok dan kalender toko.")


## 3. Pola permintaan

Bandingkan hari dalam minggu dan hitung laju jual 28 hari per produk. Laju jual adalah **penjualan tercatat**, bukan permintaan murni: kita tidak punya riwayat stok harian untuk mengetahui kapan barang tidak tersedia. Produk tanpa transaksi 28 hari ditandai untuk ditinjau, bukan langsung dianggap tidak diminati.


In [ ]:
weekday_names = ["Sen", "Sel", "Rab", "Kam", "Jum", "Sab", "Min"]
calendar_days = pd.date_range(transactions.transaction_date.min(), transactions.transaction_date.max(), freq="D")
days_per_weekday = pd.Series(calendar_days.weekday).value_counts().reindex(range(7), fill_value=0)
weekday_sales = sales.groupby(sales.transaction_date.dt.weekday).agg(
    units_sold=("quantity", "sum"), revenue=("revenue", "sum")
).reindex(range(7), fill_value=0)
weekday_sales["days_observed"] = days_per_weekday.to_numpy()
weekday_sales["avg_units_per_day"] = weekday_sales.units_sold / weekday_sales.days_observed
weekday_sales.index = weekday_names
print(weekday_sales.round(1).to_string())

recent_start = snapshot - pd.Timedelta(days=28)
recent_sales = sales.loc[
    (sales.transaction_date >= recent_start) & (sales.transaction_date < snapshot)
].groupby("product_id").agg(
    units_sold_28d=("quantity", "sum"), revenue_28d=("revenue", "sum")
)
print(f"\nPeriode terbaru: {recent_start.date()} s.d. {(snapshot - pd.Timedelta(days=1)).date()}")
print(f"Produk dengan penjualan tercatat: {len(recent_sales):,}")


In [ ]:
# Grafik 4–6: pola hari, sebaran penjualan SKU, dan kategori terbaru.
stocked_recent = products.loc[products.stock_on_hand > 0, ["product_id", "category"]].copy()
stocked_recent = stocked_recent.join(recent_sales[["units_sold_28d"]], on="product_id")
stocked_recent["units_sold_28d"] = stocked_recent.units_sold_28d.fillna(0)
recent_bins = pd.cut(
    stocked_recent.units_sold_28d, bins=[-1, 0, 1, 3, 7, np.inf],
    labels=["0", "1", "2–3", "4–7", "8+"]
).value_counts().reindex(["0", "1", "2–3", "4–7", "8+"], fill_value=0)
recent_category = sales.loc[
    (sales.transaction_date >= recent_start) & (sales.transaction_date < snapshot)
].groupby("category").quantity.sum().sort_values(ascending=False).head(10).sort_values()
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    axes[0].bar(weekday_sales.index, weekday_sales.avg_units_per_day, color="#367c8d")
    axes[0].set(title="Rata-rata unit terjual per hari", ylabel="Unit per hari")
    axes[1].bar(recent_bins.index.astype(str), recent_bins.values, color="#e0a13f")
    axes[1].set(title="Produk berstok menurut unit terjual 28 hari", xlabel="Unit terjual per produk", ylabel="Jumlah produk")
    axes[2].barh(recent_category.index, recent_category.values, color="#74a892")
    axes[2].set(title="Kategori terlaris dalam 28 hari", xlabel="Unit")
    for ax in axes:
        ax.grid(axis="y" if ax is not axes[2] else "x", alpha=.2)
    fig.tight_layout()
    plt.show()
peak_day = weekday_sales.avg_units_per_day.idxmax()
print(f"Insight: {peak_day} mencatat rata-rata unit per hari tertinggi. "
      f"Sebanyak {recent_bins.loc['0']:,} produk berstok tidak mencatat penjualan 28 hari terakhir; "
      "cek ketersediaan dan penempatan produk sebelum menyebutnya tidak diminati.")


## 4. Harga, diskon, dan margin

Ringkas apa yang *tercatat* pada tiap rentang diskon. Perbedaan penjualan antar-rentang adalah **asosiasi**, bukan bukti kenaikan akibat diskon: produk, hari, event, dan ketersediaan stok dapat berbeda. Margin kotor di sini memakai `unit_cost` katalog saat ini sebagai pendekatan; riwayat biaya per transaksi belum tersedia.

**Khusus benchmark:** backend menyimpan jendela diskon, tetapi tidak menyimpan `unit_list_price` atau `discount_rate` pada tiap transaksi. Grafik band diskon ini belum dapat dihitung dengan kontrak backend saat ini.


In [ ]:
discount_bins = [-0.001, 0.000001, 0.05, 0.10, 0.20, 1.01]
discount_labels = ["0%", ">0–5%", ">5–10%", ">10–20%", ">20%"]
sales["discount_band"] = pd.cut(sales.discount_rate, bins=discount_bins, labels=discount_labels)
sales["gross_profit_proxy"] = sales.quantity * (sales.unit_sale_price - sales.unit_cost)
discount_summary = sales.groupby("discount_band", observed=True).agg(
    units_sold=("quantity", "sum"), revenue=("revenue", "sum"),
    avg_sale_price=("unit_sale_price", "mean"), gross_profit_proxy=("gross_profit_proxy", "sum")
)
discount_summary["gross_margin_proxy"] = discount_summary.gross_profit_proxy / discount_summary.revenue
print(discount_summary.round({"avg_sale_price": 0, "gross_margin_proxy": 3}).to_string())
print("\nCatatan: band diskon bisa memiliki campuran produk dan periode yang berbeda.")


In [ ]:
# Grafik 7–9: campuran tingkat diskon, margin historis, dan jenis promosi.
promo_counts = sales.promotion_type.value_counts().sort_values()
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    axes[0].bar(discount_summary.index.astype(str), discount_summary.units_sold / discount_summary.units_sold.sum() * 100,
                color="#367c8d")
    axes[0].set(title="Porsi unit terjual per tingkat diskon", ylabel="Persen dari unit terjual")
    axes[1].bar(discount_summary.index.astype(str), discount_summary.gross_margin_proxy * 100,
                color="#e07a3f")
    axes[1].axhline(0, color="#555555", linewidth=.8)
    axes[1].set(title="Margin kotor pendekatan", ylabel="Persen dari omzet")
    axes[2].barh(promo_counts.index, promo_counts.values, color="#74a892")
    axes[2].set(title="Jumlah transaksi per jenis promosi", xlabel="Transaksi")
    for ax in axes[:2]:
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=.2)
    axes[2].grid(axis="x", alpha=.2)
    fig.tight_layout()
    plt.show()
print("Insight: margin per band diskon terlihat berbeda, tetapi campuran produk dan waktu juga berbeda. "
      "Grafik ini belum mengukur tambahan penjualan akibat diskon.")


## 5. Risiko stok pada tanggal snapshot

Untuk setiap produk dengan stok positif, hitung `days_to_expiry`, penjualan 28 hari, dan *days of cover* (`stock_on_hand / rata-rata unit terjual per hari`). Untuk produk yang kedaluwarsa dalam **90 hari**, `units_at_risk_proxy` adalah stok dikurangi perkiraan unit yang terjual sampai tanggal kedaluwarsa **jika laju 28 hari berlanjut**. Di luar jendela itu indikator tidak dihitung karena ekstrapolasinya terlalu jauh. Ini indikator penyaringan, bukan forecast atau angka kerugian pasti. Produk yang sudah mencapai tanggal kedaluwarsa dipisahkan untuk pemeriksaan kelayakan jual, bukan dipromosikan.

**Khusus benchmark:** `expiry_date` dan `stock_as_of_date` belum menjadi field tabel backend. Prioritas berbasis kedaluwarsa di bagian ini tidak boleh langsung dipindahkan ke backend. Bagian 7 menyediakan perhitungan yang bisa dipindahkan.


In [ ]:
inventory = products.loc[products.stock_on_hand > 0, [
    "product_id", "sku", "product_name", "category", "unit_cost", "list_price", "stock_on_hand", "expiry_date"
]].copy()
inventory = inventory.join(recent_sales, on="product_id")
inventory[["units_sold_28d", "revenue_28d"]] = inventory[["units_sold_28d", "revenue_28d"]].fillna(0)
inventory["units_sold_28d"] = inventory.units_sold_28d.astype(int)
inventory["days_to_expiry"] = (inventory.expiry_date - snapshot).dt.days
inventory["daily_sales_rate_28d"] = inventory.units_sold_28d / 28
inventory["days_of_cover"] = inventory.stock_on_hand / inventory.daily_sales_rate_28d.replace(0, np.nan)
inventory["expected_units_until_expiry_proxy"] = (
    inventory.daily_sales_rate_28d * inventory.days_to_expiry.clip(lower=0)
).clip(upper=inventory.stock_on_hand)
inventory["units_at_risk_proxy"] = (
    inventory.stock_on_hand - inventory.expected_units_until_expiry_proxy
).clip(lower=0).round().astype("Int64")
inventory.loc[inventory.days_to_expiry > 90, "units_at_risk_proxy"] = pd.NA
inventory["cost_at_risk_proxy"] = inventory.units_at_risk_proxy * inventory.unit_cost
inventory["gross_margin_rate_at_list_price"] = (inventory.list_price - inventory.unit_cost) / inventory.list_price

conditions = [
    inventory.days_to_expiry <= 0,
    inventory.days_to_expiry <= 30,
    inventory.days_to_expiry <= 90,
    inventory.units_sold_28d == 0,
]
choices = ["check_expiry", "expiry_30d", "expiry_90d", "no_recent_sales"]
inventory["priority_group"] = np.select(conditions, choices, default="routine")
summary = inventory.groupby("priority_group").agg(
    products=("product_id", "count"), stock_units=("stock_on_hand", "sum"),
    units_at_risk_proxy=("units_at_risk_proxy", lambda values: values.sum(min_count=1)),
    cost_at_risk_proxy=("cost_at_risk_proxy", lambda values: values.sum(min_count=1))
).reindex(["check_expiry", "expiry_30d", "expiry_90d", "no_recent_sales", "routine"], fill_value=0)
print(summary.to_string())


In [ ]:
# Grafik 10–12: komposisi stok, modal yang berisiko, dan stok versus laju jual.
expiry_order = ["check_expiry", "expiry_30d", "expiry_90d", "no_recent_sales", "routine"]
expiry_labels = ["Cek tanggal", "≤30 hari", "31–90 hari", "Tanpa jual 28h", "Rutin"]
risk_by_category = inventory.loc[inventory.days_to_expiry <= 90].groupby("category").cost_at_risk_proxy.sum()
risk_by_category = risk_by_category.nlargest(10).sort_values()
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    bars = axes[0].bar(expiry_labels, summary.loc[expiry_order, "stock_units"] / 1_000,
                       color=["#b54c4c", "#e07a3f", "#e0a13f", "#7da7b5", "#74a892"])
    axes[0].set(title="Unit stok menurut kelompok prioritas", ylabel="Ribu unit")
    axes[0].tick_params(axis="x", rotation=25)
    axes[1].barh(risk_by_category.index, risk_by_category.values / 1_000_000, color="#b54c4c")
    axes[1].set(title="10 kategori: modal berisiko (proksi)", xlabel="Juta rupiah")
    scatter = inventory.loc[inventory.days_to_expiry > 0].sample(
        n=min(2000, (inventory.days_to_expiry > 0).sum()), random_state=42
    )
    urgent = scatter.days_to_expiry.le(30)
    axes[2].scatter(scatter.loc[~urgent, "units_sold_28d"] + 1, scatter.loc[~urgent, "stock_on_hand"] + 1,
                    alpha=.35, s=12, color="#367c8d", label=">30 hari")
    axes[2].scatter(scatter.loc[urgent, "units_sold_28d"] + 1, scatter.loc[urgent, "stock_on_hand"] + 1,
                    alpha=.65, s=15, color="#e07a3f", label="≤30 hari")
    axes[2].set(xscale="log", yscale="log", title="Stok vs penjualan 28 hari",
                xlabel="Unit terjual 28 hari + 1 (log)", ylabel="Stok unit + 1 (log)")
    axes[2].legend(frameon=False)
    for ax in axes[:2]: ax.grid(axis="y" if ax is axes[0] else "x", alpha=.2)
    fig.tight_layout()
    plt.show()
near_expiry = inventory.days_to_expiry.between(1, 30)
print(f"Insight: {near_expiry.sum():,} produk dengan {inventory.loc[near_expiry, 'stock_on_hand'].sum():,} unit "
      "stok berada dalam 30 hari menuju kedaluwarsa. Nilai risiko adalah proksi, bukan kerugian pasti.")


## 6. Daftar prioritas produk

Urutkan produk yang perlu pemeriksaan tanggal kedaluwarsa, lalu stok yang berpotensi tersisa dalam 30 dan 90 hari. Nilai risiko hanyalah pendekatan untuk *triage*. Tim toko tetap perlu memeriksa stok fisik, batch kedaluwarsa, dan kelayakan jual sebelum bertindak. Notebook **tidak** mengeluarkan besaran diskon karena hubungan kausal harga–permintaan belum diketahui.

Daftar ini memakai tanggal kedaluwarsa dari CSV benchmark. Gunakan daftar pada bagian 7 untuk kontrak backend yang tersedia sekarang.


In [ ]:
priority_order = {"check_expiry": 0, "expiry_30d": 1, "expiry_90d": 2, "no_recent_sales": 3, "routine": 4}
priority_products = inventory.assign(
    priority_rank=inventory.priority_group.map(priority_order)
).sort_values(
    ["priority_rank", "cost_at_risk_proxy", "days_to_expiry"],
    ascending=[True, False, True]
)[[
    "product_id", "sku", "product_name", "category", "priority_group", "stock_on_hand",
    "units_sold_28d", "days_of_cover", "days_to_expiry", "units_at_risk_proxy",
    "cost_at_risk_proxy", "gross_margin_rate_at_list_price"
]].reset_index(drop=True)
preview = priority_products.head(20).copy()
preview["days_of_cover"] = preview.days_of_cover.map(lambda x: "—" if pd.isna(x) else f"{x:,.0f}")
print(preview.to_string(index=False, formatters={
    "gross_margin_rate_at_list_price": "{:.1%}".format,
}))

# Jika perlu file untuk dashboard/peninjauan manual, ubah menjadi True lalu jalankan sel ini.
EXPORT_PRIORITY_CSV = False
if EXPORT_PRIORITY_CSV:
    output_path = Path.cwd() / "priority_products.csv"
    priority_products.to_csv(output_path, index=False)
    print("File tersimpan:", output_path)


In [ ]:
# Grafik 13–14: jumlah produk dan stok dalam daftar prioritas.
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].bar(expiry_labels, summary.loc[expiry_order, "products"],
                color=["#b54c4c", "#e07a3f", "#e0a13f", "#7da7b5", "#74a892"])
    axes[0].set(title="Jumlah produk per kelompok prioritas", ylabel="Produk")
    axes[0].tick_params(axis="x", rotation=25)
    urgent_products = priority_products.loc[priority_products.priority_group == "expiry_30d"].head(10)
    axes[1].barh(urgent_products.product_id.iloc[::-1],
                 urgent_products.cost_at_risk_proxy.iloc[::-1] / 1_000_000, color="#e07a3f")
    axes[1].set(title="10 produk ≤30 hari: modal berisiko (proksi)", xlabel="Juta rupiah")
    for ax in axes: ax.grid(axis="y" if ax is axes[0] else "x", alpha=.2)
    fig.tight_layout()
    plt.show()
print("Insight: mulai peninjauan dari kelompok 'check_expiry', lalu produk ≤30 hari dengan stok dan "
      "modal berisiko terbesar. Verifikasi tanggal batch dan kondisi fisik sebelum tindakan harga.")


## 7. Prioritas stok yang sesuai backend Kapanin

Backend saat ini menyediakan `products(id, name, category, cost, price)`, `stock(product_id, quantity, stock_since)`, dan `sales(product_id, quantity, price_sold, sold_on)`. CSV diproyeksikan ke nama field tersebut **di memori**; ID demo bukan UUID backend. Tidak ada CSV baru karena salinan data tidak menyelesaikan perbedaan skema. `expiry_date` dan detail band diskon sengaja tidak dipakai pada perhitungan berikut.

`as_of_date` adalah tanggal pembacaan **stok terkini** (2026-10-01 pada benchmark). Pada backend nyata, nilainya adalah tanggal saat query dijalankan. Karena belum ada riwayat snapshot stok, endpoint tidak boleh mengklaim dapat menghitung prioritas stok untuk tanggal lampau.


In [ ]:
# Proyeksi field yang tersedia di tabel backend; owner_id berasal dari JWT saat implementasi.
backend_products = products.rename(columns={
    "product_id": "id", "product_name": "name", "unit_cost": "cost", "list_price": "price"
})[["id", "name", "category", "cost", "price"]].copy()
backend_stock = products.rename(columns={
    "product_id": "product_id", "stock_on_hand": "quantity", "stock_received_date": "stock_since"
})[["product_id", "quantity", "stock_since"]].copy()
backend_sales = transactions.rename(columns={
    "transaction_id": "id", "transaction_date": "sold_on", "unit_sale_price": "price_sold"
})[["id", "product_id", "quantity", "price_sold", "sold_on"]].copy()
backend_as_of_date = products.stock_as_of_date.max().normalize()
assert products.stock_as_of_date.eq(backend_as_of_date).all(), "Snapshot stok harus satu tanggal"
assert backend_products.id.is_unique
assert backend_stock.product_id.isin(backend_products.id).all()
assert backend_sales.product_id.isin(backend_products.id).all()
print("Proyeksi backend:", len(backend_products), "produk,", len(backend_stock), "baris stok,",
      len(backend_sales), "penjualan; snapshot", backend_as_of_date.date())


In [ ]:
def build_stock_priorities(product_rows, stock_rows, sale_rows, as_of_date, lookback_days=28):
    """Prioritas informatif dari field backend; tidak menghasilkan diskon."""
    if lookback_days <= 0:
        raise ValueError("lookback_days harus positif")
    as_of = pd.Timestamp(as_of_date).normalize()
    p = product_rows[["id", "name", "category", "cost", "price"]].copy()
    s = stock_rows[["product_id", "quantity"]].copy()
    sales_input = sale_rows[["product_id", "quantity", "sold_on"]].copy()
    sales_input["sold_on"] = pd.to_datetime(sales_input.sold_on).dt.normalize()
    if not p.id.is_unique or p.id.isna().any():
        raise ValueError("ID produk harus unik dan terisi")
    if (s.quantity < 0).any() or (sales_input.quantity < 0).any():
        raise ValueError("Kuantitas stok/penjualan tidak valid")
    if (p.cost < 0).any() or (p.price < 0).any():
        raise ValueError("Harga/biaya tidak valid")
    if not s.product_id.isin(p.id).all() or not sales_input.product_id.isin(p.id).all():
        raise ValueError("Relasi product_id tidak valid")
    window_start = as_of - pd.Timedelta(days=lookback_days)
    recent = sales_input.loc[sales_input.sold_on.ge(window_start) & sales_input.sold_on.lt(as_of)]
    stock_total = s.groupby("product_id").quantity.sum().rename("stock_quantity")
    units = recent.groupby("product_id").quantity.sum().rename("units_sold_28d")
    result = p.join(stock_total, on="id").join(units, on="id")
    result["stock_quantity"] = result.stock_quantity.fillna(0).astype(int)
    result["units_sold_28d"] = result.units_sold_28d.fillna(0).astype(int)
    result = result.loc[result.stock_quantity.gt(0)].copy()
    result["days_of_cover"] = (
        result.stock_quantity * lookback_days / result.units_sold_28d.replace(0, np.nan)
    )
    result["remaining_28d_proxy"] = (result.stock_quantity - result.units_sold_28d).clip(lower=0)
    result["tied_cost_28d_proxy"] = result.remaining_28d_proxy * result.cost
    result["priority_group"] = np.select(
        [result.units_sold_28d.eq(0), result.days_of_cover.gt(90), result.days_of_cover.gt(30)],
        ["no_sales_28d", "cover_gt_90d", "cover_30_90d"], default="cover_le_30d"
    )
    order = {"no_sales_28d": 0, "cover_gt_90d": 1, "cover_30_90d": 2, "cover_le_30d": 3}
    result["priority_rank"] = result.priority_group.map(order)
    return result.sort_values(
        ["priority_rank", "tied_cost_28d_proxy", "id"], ascending=[True, False, True]
    ).reset_index(drop=True)

backend_priority = build_stock_priorities(
    backend_products, backend_stock, backend_sales, backend_as_of_date
)
backend_priority_summary = backend_priority.groupby("priority_group").agg(
    products=("id", "count"), stock_units=("stock_quantity", "sum"),
    remaining_28d_proxy=("remaining_28d_proxy", "sum"),
    tied_cost_28d_proxy=("tied_cost_28d_proxy", "sum")
).reindex(["no_sales_28d", "cover_gt_90d", "cover_30_90d", "cover_le_30d"], fill_value=0)
print(backend_priority_summary.to_string())
print("\nContoh 10 produk untuk peninjauan:")
print(backend_priority[["id", "name", "priority_group", "stock_quantity", "units_sold_28d",
                        "days_of_cover", "remaining_28d_proxy", "tied_cost_28d_proxy"]].head(10).round(1).to_string(index=False))


In [ ]:
# Grafik backend-ready: jumlah produk dan modal tersisa pada laju jual 28 hari.
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    labels = ["Tanpa jual 28h", ">90 hari", "30–90 hari", "≤30 hari"]
    colors = ["#b54c4c", "#e07a3f", "#e0a13f", "#74a892"]
    axes[0].bar(labels, backend_priority_summary.products, color=colors)
    axes[0].set(title="Produk berstok menurut laju jual", ylabel="Jumlah produk")
    axes[1].bar(labels, backend_priority_summary.tied_cost_28d_proxy / 1_000_000, color=colors)
    axes[1].set(title="Modal tersisa 28 hari (proksi)", ylabel="Juta rupiah")
    for ax in axes:
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=.2)
    fig.tight_layout()
    plt.show()


In [ ]:
# Pemeriksaan batas periode, stok multi-baris, dan penjualan bernilai nol.
fixture_products = pd.DataFrame({
    "id": ["A", "B"], "name": ["A", "B"], "category": ["demo", "demo"],
    "cost": [10, 20], "price": [15, 30]
})
fixture_stock = pd.DataFrame({"product_id": ["A", "A", "B"], "quantity": [3, 4, 5]})
fixture_sales = pd.DataFrame({
    "product_id": ["A", "A", "A", "B"], "quantity": [9, 2, 5, 0],
    "sold_on": ["2026-09-02", "2026-09-03", "2026-10-01", "2026-09-10"]
})
fixture_result = build_stock_priorities(
    fixture_products, fixture_stock, fixture_sales, "2026-10-01"
).set_index("id")
assert fixture_result.loc["A", "stock_quantity"] == 7
assert fixture_result.loc["A", "units_sold_28d"] == 2
assert fixture_result.loc["A", "days_of_cover"] == 98
assert fixture_result.loc["B", "priority_group"] == "no_sales_28d"
assert backend_priority.stock_quantity.sum() == backend_stock.quantity.sum()
print("Pemeriksaan kontrak backend lulus: batas 28 hari, agregasi stok, dan nol penjualan.")


### Kontrak untuk tahap backend

Endpoint baca yang dapat dibuat berikutnya: `GET /analytics/stock-priorities`. Identitas pemilik berasal dari JWT seperti endpoint lain. Query mengambil produk, total stok terkini, dan penjualan pada `[tanggal_baca - 28 hari, tanggal_baca)` milik pemilik itu saja. Respons per produk memuat `product_id`, `stock_quantity`, `units_sold_28d`, `days_of_cover` (null bila tidak ada penjualan), `remaining_28d_proxy`, `tied_cost_28d_proxy`, dan `priority_group`. Urutan mengikuti fungsi di atas. Nilai 30/90 hari adalah ambang triage prototipe yang perlu disepakati dengan pemilik toko.

`remaining_28d_proxy` mengasumsikan laju jual 28 hari terulang satu kali. Ia bukan prediksi terkalibrasi atau jumlah barang yang pasti kedaluwarsa. Penjualan nol bisa disebabkan stok historis tidak tersedia. Untuk rekomendasi kedaluwarsa, backend perlu tanggal per batch dan riwayat stok; untuk besaran diskon, perlu riwayat harga/promo yang dapat dibandingkan dan evaluasi efeknya.


## 8. Training prediksi permintaan

**Alur dari EDA ke model:** bagian 1 memvalidasi `products` dan `transactions`; bagian 2 membentuk `sales` berisi transaksi dengan kategori produk. Bagian ini memakai **variabel `sales` yang sama** untuk membuat target kategori × minggu, fitur lag, model, dan backtest. Tidak ada pembacaan CSV kedua. Jalankan notebook dari sel pertama agar `sales` dan tanggal snapshot tersedia.

Target awal adalah unit terjual **per kategori pada minggu berikutnya**. Prediksi ini menjadi konteks permintaan untuk peninjauan stok, belum menentukan waktu atau besaran diskon produk. Model hanya boleh dipakai bila mengalahkan baseline pada periode yang belum dilihat.

Pandas dan NumPy cukup untuk Ridge dan EWMA. Jika scikit-learn atau LightGBM tersedia, kandidat model pohon ikut dilatih. Di Colab, jalankan `%pip install scikit-learn lightgbm` pada sel terpisah bila paket belum tersedia, lalu jalankan ulang mulai sel pemeriksaan paket di bawah.


In [ ]:
try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False
print("Model pohon scikit-learn tersedia:", HAS_SKLEARN, "| LightGBM tersedia:", HAS_LIGHTGBM)


### 8.1 Target dan periode evaluasi

Satu baris = satu kategori × satu minggu penuh (Senin–Minggu); target = total unit terjual minggu itu. Minggu pertama dan terakhir yang parsial dibuang. Minggu tanpa transaksi dicatat nol **penjualan tercatat**, bukan klaim bahwa permintaannya nol: riwayat stok harian belum tersedia. Kita memakai tahun 2023 hanya sebagai riwayat lag dan mulai melatih dari 2024.

- **Train:** 2024–2025.
- **Validasi:** Januari–Juni 2026, untuk memilih model.
- **Test:** Juli–September 2026, hanya untuk evaluasi akhir.

Setiap baris hanya memakai informasi yang sudah tersedia sebelum minggu target. Prediksi ini adalah **satu minggu ke depan**; pada backtest, minggu berikutnya boleh memakai hasil aktual minggu sebelumnya setelah hasil itu terjadi.


In [ ]:
sales["week_start"] = sales.transaction_date - pd.to_timedelta(sales.transaction_date.dt.weekday, unit="D")
first_date, last_date = sales.transaction_date.min(), sales.transaction_date.max()
first_full_monday = first_date + pd.Timedelta(days=(-first_date.weekday()) % 7)
last_full_monday = last_date - pd.Timedelta(days=last_date.weekday()+7) if last_date.weekday() != 6 else last_date-pd.Timedelta(days=6)
weeks = pd.date_range(first_full_monday,last_full_monday,freq="W-MON")
category_index = pd.MultiIndex.from_product([sorted(products.category.dropna().unique()),weeks],names=["category","week_start"])
weekly = sales.groupby(["category","week_start"]).quantity.sum().reindex(category_index,fill_value=0).rename("units_sold").reset_index()
weekly = weekly.sort_values(["category","week_start"]).reset_index(drop=True)
g = weekly.groupby("category",sort=False).units_sold
weekly["lag_1"] = g.shift(1)
weekly["mean_4"] = g.transform(lambda s: s.shift(1).rolling(4,min_periods=4).mean())
weekly["mean_12"] = g.transform(lambda s: s.shift(1).rolling(12,min_periods=12).mean())
weekly["lag_52"] = g.shift(52)
weekly["week_index"] = (weekly.week_start-weeks[0]).dt.days/7
week_no = weekly.week_start.dt.isocalendar().week.astype(int)
weekly["week_sin"] = np.sin(2*np.pi*week_no/52.18)
weekly["week_cos"] = np.cos(2*np.pi*week_no/52.18)
# Hitung EWMA sebelum membuang 2023 agar awal 2024 memiliki riwayat yang benar.
for alpha in (0.1,0.2,0.3,0.4,0.5,0.7):
    weekly[f"ewma_{alpha}"] = weekly.groupby("category",sort=False).units_sold.transform(
        lambda series: series.shift(1).ewm(alpha=alpha,adjust=False).mean()
    )
weekly = weekly.loc[weekly.week_start.ge(pd.Timestamp("2024-01-01"))].dropna().reset_index(drop=True)
train_mask = weekly.week_start.lt(pd.Timestamp("2026-01-01")).to_numpy()
val_mask = weekly.week_start.between("2026-01-01","2026-06-30").to_numpy()
test_mask = weekly.week_start.ge(pd.Timestamp("2026-07-01")).to_numpy()
assert train_mask.any() and val_mask.any() and test_mask.any()
assert weekly.loc[train_mask,"week_start"].max() < weekly.loc[val_mask,"week_start"].min()
assert weekly.loc[val_mask,"week_start"].max() < weekly.loc[test_mask,"week_start"].min()
# Pemeriksaan contoh: fitur lag harus sama dengan target pada minggu sebelumnya.
sample_category = weekly.category.iloc[0]
sample_history = weekly.loc[weekly.category.eq(sample_category)].reset_index(drop=True)
assert sample_history.loc[20,"lag_1"] == sample_history.loc[19,"units_sold"]
assert np.isclose(sample_history.loc[20,"mean_4"],sample_history.loc[16:19,"units_sold"].mean())
print("Minggu penuh:",len(weeks),"| baris model:",len(weekly))
print("Train/validasi/test:",train_mask.sum(),val_mask.sum(),test_mask.sum())
print("Porsi target nol:",round(weekly.units_sold.eq(0).mean(),3))


### 8.2 Fitur dan baseline

Fitur: penjualan kategori minggu lalu, rata-rata 4 dan 12 minggu sebelumnya, minggu yang sama tahun sebelumnya, posisi waktu, serta musim kalender. Identitas kategori dikodekan sebagai kolom indikator. Tidak ada harga/diskon minggu target, stok snapshot, atau hasil masa depan dalam fitur.

Baseline yang wajib dikalahkan: `lag_1`, `mean_4`, dan `mean_12`. Metrik utama **MAE** (rata-rata selisih unit per kategori-minggu); WAPE dan bias membantu membaca skala kesalahan dan arah perkiraan. Produk dengan jumlah unit sangat kecil dapat dipelajari terpisah setelah data stok historis tersedia.


In [ ]:
numeric_features = ["lag_1","mean_4","mean_12","lag_52","week_index","week_sin","week_cos"]
category_features = pd.get_dummies(weekly.category,prefix="category",dtype=float)
feature_names = numeric_features + category_features.columns.tolist()
X_raw = np.column_stack([weekly[numeric_features].to_numpy(dtype=float),category_features.to_numpy(dtype=float)])
y = weekly.units_sold.to_numpy(dtype=float)
feature_mean = X_raw[train_mask].mean(axis=0)
feature_std = X_raw[train_mask].std(axis=0)
feature_std[feature_std==0] = 1
X = (X_raw-feature_mean)/feature_std
X_with_intercept = np.column_stack([np.ones(len(X)),X])

def metrics(actual,predicted):
    error = np.asarray(predicted)-np.asarray(actual)
    return {"MAE":float(np.abs(error).mean()),
            "WAPE":float(np.abs(error).sum()/np.asarray(actual).sum()) if np.asarray(actual).sum() else np.nan,
            "bias_units":float(error.mean())}

baseline_predictions = {name:weekly[name].to_numpy(dtype=float) for name in ("lag_1","mean_4","mean_12")}
print("Fitur:",len(feature_names),"| baseline:",list(baseline_predictions))


### 8.3 Training kandidat

Ridge dan EWMA berjalan dengan NumPy/Pandas. EWMA memiliki satu parameter, `alpha`: makin besar nilainya, makin kuat bobot minggu terbaru. Nilai alpha dan regularisasi Ridge dipilih memakai validasi. Jika tersedia, model pohon Poisson juga dilatih memakai **train saja**. Semua kandidat menerima riwayat yang sama; test tidak dipakai untuk tuning.


In [ ]:
def fit_ridge(x_train,y_train,alpha):
    penalty = np.eye(x_train.shape[1])*alpha
    penalty[0,0] = 0  # intercept tidak diregularisasi
    return np.linalg.solve(x_train.T@x_train+penalty,x_train.T@y_train)

def predict_ridge(x,coef):
    return np.clip(x@coef,0,None)

ridge_trials = []
for alpha in (0.1,1,10,100,1000,10000):
    coef = fit_ridge(X_with_intercept[train_mask],y[train_mask],alpha)
    pred = predict_ridge(X_with_intercept[val_mask],coef)
    ridge_trials.append((alpha,metrics(y[val_mask],pred)["MAE"]))
best_alpha = min(ridge_trials,key=lambda item:item[1])[0]
ridge_coef = fit_ridge(X_with_intercept[train_mask],y[train_mask],best_alpha)
model_predictions = {"ridge":predict_ridge(X_with_intercept,ridge_coef)}
print("Ridge alpha terpilih:",best_alpha)

# Satu langkah ke depan: EWMA pada minggu t hanya melihat penjualan sampai t-1.
ewma_trials = []
for alpha in (0.1,0.2,0.3,0.4,0.5,0.7):
    pred = weekly[f"ewma_{alpha}"].to_numpy(dtype=float)
    assert np.isfinite(pred).all()
    ewma_trials.append((alpha,metrics(y[val_mask],pred[val_mask])["MAE"],pred))
best_ewma_alpha,best_ewma_mae,best_ewma_pred = min(ewma_trials,key=lambda item:item[1])
model_predictions["ewma_tuned"] = best_ewma_pred
print("EWMA alpha terpilih:",best_ewma_alpha,"| MAE validasi:",round(best_ewma_mae,2))

if HAS_SKLEARN:
    tree_model = HistGradientBoostingRegressor(
        loss="poisson",learning_rate=0.05,max_iter=200,max_leaf_nodes=15,
        min_samples_leaf=30,l2_regularization=10,early_stopping=False,random_state=42
    )
    tree_model.fit(X[train_mask],y[train_mask])
    model_predictions["hist_gradient_boosting"] = np.clip(tree_model.predict(X),0,None)
    print("HistGradientBoosting berhasil dilatih.")
if HAS_LIGHTGBM:
    lightgbm_model = LGBMRegressor(
        objective="poisson",n_estimators=200,learning_rate=0.05,num_leaves=15,
        min_child_samples=30,reg_lambda=10,random_state=42,verbosity=-1
    )
    lightgbm_model.fit(X_raw[train_mask],y[train_mask])
    model_predictions["lightgbm_demand"] = np.clip(lightgbm_model.predict(X_raw),0,None)
    print("LightGBM demand berhasil dilatih.")


### 8.4 Backtest dan keputusan

Pilih kandidat berdasarkan MAE validasi saja, kemudian periksa test. Suatu model baru hanya layak menjadi **calon** penggunaan jika MAE-nya minimal **5% lebih rendah** dari baseline terbaik pada validasi dan dari semua baseline yang telah ditetapkan pada test. Ambang 5% adalah aturan keputusan awal; ia bukan bukti dampak bisnis. Jika syarat gagal, keluaran tetap analisis eksperimen.


In [ ]:
all_predictions = {**baseline_predictions,**model_predictions}
rows=[]
for name,pred in all_predictions.items():
    for split,mask in (("validation",val_mask),("test",test_mask)):
        rows.append({"candidate":name,"split":split,**metrics(y[mask],pred[mask])})
scorecard = pd.DataFrame(rows).sort_values(["split","MAE","candidate"]).reset_index(drop=True)
print(scorecard.to_string(index=False,formatters={"MAE":"{:.2f}".format,"WAPE":"{:.1%}".format,"bias_units":"{:+.2f}".format}))
validation_scores = scorecard.loc[scorecard.split.eq("validation")].set_index("candidate")
test_scores = scorecard.loc[scorecard.split.eq("test")].set_index("candidate")
selected_candidate = validation_scores.MAE.idxmin()
baseline_names = ["lag_1","mean_4","mean_12"]
best_baseline_validation = validation_scores.loc[baseline_names,"MAE"].idxmin()
best_baseline_test = test_scores.loc[baseline_names,"MAE"].idxmin()
validation_gain = 1-validation_scores.loc[selected_candidate,"MAE"]/validation_scores.loc[best_baseline_validation,"MAE"]
test_gain = 1-test_scores.loc[selected_candidate,"MAE"]/test_scores.loc[best_baseline_test,"MAE"]
MIN_RELATIVE_GAIN = 0.05
model_ready = (selected_candidate in model_predictions and
               validation_gain >= MIN_RELATIVE_GAIN and test_gain >= MIN_RELATIVE_GAIN)
print("\nTerpilih dari validasi:",selected_candidate)
print("Baseline validasi:",best_baseline_validation,"| baseline test paling kuat:",best_baseline_test)
print(f"Keunggulan MAE vs baseline: validasi {validation_gain:.1%}; test {test_gain:.1%}")
print("Lolos gerbang 5% sebagai calon model:",model_ready)
if not model_ready:
    print("Belum ada model yang cukup kuat untuk dipakai di API. Pertahankan baseline sebagai pembanding.")


In [ ]:
# Grafik: akurasi kandidat, total volume test, dan kesalahan per kategori.
selected_pred = all_predictions[selected_candidate]
test_detail = weekly.loc[test_mask,["category","week_start"]].copy()
test_detail["actual"] = y[test_mask]
test_detail["selected_pred"] = selected_pred[test_mask]
test_detail["baseline_pred"] = all_predictions[best_baseline_test][test_mask]
test_detail["selected_abs_error"] = (test_detail.selected_pred-test_detail.actual).abs()
test_detail["baseline_abs_error"] = (test_detail.baseline_pred-test_detail.actual).abs()
category_errors = test_detail.groupby("category").agg(
    actual_units=("actual","sum"), selected_mae=("selected_abs_error","mean"),
    baseline_mae=("baseline_abs_error","mean")
).sort_values("selected_mae",ascending=False)
print("\n10 kategori dengan kesalahan terbesar pada test:")
print(category_errors.head(10).round(2).to_string())

if HAS_PLOTS:
    fig,axes=plt.subplots(1,3,figsize=(18,4.5))
    test_rows=scorecard.loc[scorecard.split.eq("test")].sort_values("MAE")
    axes[0].barh(test_rows.candidate.iloc[::-1],test_rows.MAE.iloc[::-1],color="#367c8d")
    axes[0].set(title="MAE test per kategori-minggu",xlabel="Unit; lebih kecil lebih baik")
    total=test_detail.groupby("week_start")[["actual","selected_pred","baseline_pred"]].sum()
    axes[1].plot(total.index,total.actual,label="Aktual",color="#0b6e75")
    axes[1].plot(total.index,total.selected_pred,label=selected_candidate,color="#e07a3f")
    axes[1].plot(total.index,total.baseline_pred,label=best_baseline_test,color="#9a7dba",linestyle="--")
    axes[1].set(title="Total unit mingguan pada test",ylabel="Unit")
    axes[1].legend(frameon=False)
    axes[1].tick_params(axis="x",rotation=30)
    worst=category_errors.head(10).iloc[::-1]
    axes[2].barh(worst.index,worst.selected_mae,color="#b54c4c")
    axes[2].set(title="10 kategori tersulit pada test",xlabel="MAE unit per minggu")
    fig.tight_layout()
    plt.show()


### 8.5 Stabilitas dan keterbatasan hasil

Periksa apakah kesalahan lebih besar pada kategori tertentu dan apakah rata-rata volume bergeser antarperiode. Interval di bawah memakai resampling **minggu test** untuk selisih MAE kandidat terpilih vs baseline test terkuat. Ini pemeriksaan kestabilan eksploratif, bukan interval kausal atau jaminan performa produksi.


In [ ]:
volume_by_split = pd.DataFrame({
    "split":["train","validation","test"],
    "avg_units_per_category_week":[y[m].mean() for m in (train_mask,val_mask,test_mask)],
    "zero_rate":[(y[m]==0).mean() for m in (train_mask,val_mask,test_mask)]
})
print(volume_by_split.round(3).to_string(index=False))
volume_change = volume_by_split.loc[2,"avg_units_per_category_week"] / volume_by_split.loc[0,"avg_units_per_category_week"] - 1
print(f"Perubahan rata-rata volume kategori-minggu train → test: {volume_change:+.1%}")
weekly_error = test_detail.groupby("week_start")[["selected_abs_error","baseline_abs_error"]].mean()
weekly_gain = (weekly_error.baseline_abs_error-weekly_error.selected_abs_error).to_numpy()
rng = np.random.default_rng(42)
bootstrap = rng.choice(weekly_gain,size=(2000,len(weekly_gain)),replace=True).mean(axis=1)
lo,hi = np.quantile(bootstrap,[0.025,0.975])
print(f"Selisih MAE mingguan baseline − kandidat: {weekly_gain.mean():+.3f} unit; "
      f"interval bootstrap eksploratif 95% [{lo:+.3f}, {hi:+.3f}]")


### 8.6 Kaitan ke diagram model Kapanin

- **Waktu:** eksperimen kategori mingguan ini adalah pembanding terlatih sebelum mencoba TFT. TFT baru layak dibandingkan jika sinyal mingguan, fitur stok historis, dan hasil backtest mendukung kompleksitasnya.
- **Urgency:** jangan melatih regressor untuk meniru skor aturan. Perlu target bisnis yang teramati, misalnya unit yang kedaluwarsa/terbuang atau kerugian stok per periode, dengan riwayat stok dan batch.
- **Besaran diskon / T-Learner:** belum dilatih. Data perlu panel produk-hari/minggu yang mencatat periode **tanpa dan dengan promo**, stok tersedia, harga reguler dan harga jual, persentase diskon yang benar-benar diterapkan, serta outcome termasuk periode tanpa penjualan. Tanpa itu uplift tidak dapat dievaluasi dengan kredibel.

Kesimpulan notebook adalah **hasil backtest**, bukan klaim bahwa model meningkatkan penjualan atau laba. Setelah data operasional tersedia, ulangi eksperimen dengan beberapa jendela waktu dan evaluasi pada toko nyata.

Rincian target, data operasional, dan gerbang evaluasi untuk model berikutnya ada di [rencana training](../docs/model-training-plan.md).


## Kesimpulan dan langkah berikutnya

Bagian 1–6 memberi EDA benchmark sintetis. Bagian 7 menghasilkan `backend_priority` dan `backend_priority_summary` dari field yang tersedia pada backend. Bagian 8 melatih kandidat prediksi kategori-mingguan memakai `sales` dari EDA dan menghasilkan `scorecard`, `category_errors`, serta `model_ready`.

Jika `model_ready` bernilai `False`, pertahankan baseline untuk eksperimen berikutnya; jangan keluarkan prediksi model sebagai rekomendasi diskon. Prioritas stok tetap informatif dan perlu pemeriksaan fisik. Untuk tahap urgency, TFT, dan uplift/T-Learner, ikuti [rencana training](../docs/model-training-plan.md): kumpulkan riwayat stok/batch, harga reguler saat transaksi, promo yang benar-benar diterapkan, dan periode pembanding tanpa promo. Evaluasi pada toko nyata sebelum integrasi API.
